## Simple Document Processing Pipeline

This notebook section shows three separate steps:
1. Load multi-format documents (txt, md, csv, json, etc.)
2. Clean text
3. Chunk text

In [28]:
from pathlib import Path
from typing import Dict, List
import csv
import json


DEPARTMENT_ROLE_MAP: Dict[str, List[str]] = {
    "engineering": ["engineering", "tech_lead", "security", "admin"],
    "finance": ["finance", "finance_lead", "leadership", "admin"],
    "general": ["all_employees", "leadership", "admin"],
    "hr": ["hr", "hr_lead", "leadership", "admin"],
    "marketing": ["marketing", "sales", "leadership", "admin"],
}


def _read_file_as_text(file_path: Path) -> str:
    """Read different file types and normalize them to text."""
    suffix = file_path.suffix.lower()

    if suffix in {".txt", ".md", ".log", ".py", ".sql"}:
        return file_path.read_text(encoding="utf-8", errors="ignore")

    if suffix == ".csv":
        with file_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
            reader = csv.reader(f)
            rows = [", ".join(row) for row in reader]
        return "\n".join(rows)

    if suffix == ".json":
        with file_path.open("r", encoding="utf-8", errors="ignore") as f:
            obj = json.load(f)
        return json.dumps(obj, ensure_ascii=False, indent=2)

    return file_path.read_text(encoding="utf-8", errors="ignore")


def _infer_department(base: Path, file_path: Path) -> str:
    """Infer department from first folder under the configured data root."""
    try:
        rel = file_path.relative_to(base)
        if rel.parts:
            return rel.parts[0].lower()
    except ValueError:
        pass
    return "general"


def load_documents(folder: str = "data", recursive: bool = True) -> List[dict]:
    """Load files from a folder and return content with role-aware metadata."""
    base = Path(folder)

    if not base.exists() and not base.is_absolute():
        alt = Path("..") / folder
        if alt.exists():
            base = alt

    if not base.exists():
        print(f"Folder not found: {base.resolve()}")
        return []

    files = base.rglob("*") if recursive else base.glob("*")

    docs = []
    skipped = 0
    for file_path in files:
        if not file_path.is_file():
            continue

        try:
            content = _read_file_as_text(file_path)
            if not content.strip():
                continue

            department = _infer_department(base, file_path)
            docs.append(
                {
                    "source_document": file_path.name,
                    "source_path": str(file_path.resolve()),
                    "department": department,
                    "accessible_roles": DEPARTMENT_ROLE_MAP.get(
                        department, ["leadership", "admin"]
                    ),
                    "content": content,
                }
            )
        except Exception:
            skipped += 1

    print(f"Loaded {len(docs)} document(s) from {base.resolve()}")
    if skipped:
        print(f"Skipped {skipped} file(s) that could not be parsed.")
    return docs


raw_docs = load_documents("data", recursive=True)
raw_docs[:1]

Loaded 10 document(s) from D:\Projects\RBAC\data


[{'source_document': 'engineering_master_doc.md',
  'source_path': 'D:\\Projects\\RBAC\\data\\engineering\\engineering_master_doc.md',
  'department': 'engineering',
  'accessible_roles': ['engineering', 'tech_lead', 'security', 'admin'],
  'content': '# FinSolve Technologies Engineering Document\n\n## 1. Introduction\n\n### 1.1 Company Overview\nFinSolve Technologies is a leading FinTech company headquartered in Bangalore, India, with operations across North America, Europe, and Asia-Pacific. Founded in 2018, FinSolve provides innovative financial solutions, including digital banking, payment processing, wealth management, and enterprise financial analytics, serving over 2 million individual users and 10,000 businesses globally.\n\n### 1.2 Purpose\nThis engineering document outlines the technical architecture, development processes, and operational guidelines for FinSolve\'s product ecosystem. It serves as a comprehensive guide for engineering teams, stakeholders, and partners to ensu

In [29]:
import re


def clean_text(text: str) -> str:
    """Basic text cleaning: lowercase, remove extra spaces and non-word noise."""
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9.,!?;:()\-\s]", "", text)
    return text.strip()


clean_docs = []
for doc in raw_docs:
    clean_docs.append(
        {
            **doc,
            "clean_content": clean_text(doc["content"]),
        }
    )

clean_docs[:1]

[{'source_document': 'engineering_master_doc.md',
  'source_path': 'D:\\Projects\\RBAC\\data\\engineering\\engineering_master_doc.md',
  'department': 'engineering',
  'accessible_roles': ['engineering', 'tech_lead', 'security', 'admin'],
  'content': '# FinSolve Technologies Engineering Document\n\n## 1. Introduction\n\n### 1.1 Company Overview\nFinSolve Technologies is a leading FinTech company headquartered in Bangalore, India, with operations across North America, Europe, and Asia-Pacific. Founded in 2018, FinSolve provides innovative financial solutions, including digital banking, payment processing, wealth management, and enterprise financial analytics, serving over 2 million individual users and 10,000 businesses globally.\n\n### 1.2 Purpose\nThis engineering document outlines the technical architecture, development processes, and operational guidelines for FinSolve\'s product ecosystem. It serves as a comprehensive guide for engineering teams, stakeholders, and partners to ensu

In [30]:
from typing import List


def chunk_text(text: str, chunk_size: int = 400, overlap: int = 50) -> List[str]:
    """Split text into overlapping character-based chunks."""
    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and < chunk_size")

    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(text), step):
        chunk = text[i : i + chunk_size]
        if chunk:
            chunks.append(chunk)
    return chunks

In [31]:
from pathlib import Path
import csv
import json


all_chunks = []
chunk_records = []

for doc in clean_docs:
    chunks = chunk_text(doc["clean_content"], chunk_size=400, overlap=50)
    for idx, chunk in enumerate(chunks, start=1):
        chunk_id = f"CHUNK-{len(chunk_records) + 1:06d}"
        all_chunks.append(chunk)
        chunk_records.append(
            {
                "chunk_id": chunk_id,
                "source_document": doc["source_document"],
                "source_path": doc["source_path"],
                "department": doc["department"],
                "doc_type": Path(doc["source_document"]).suffix.lstrip(".").lower(),
                "chunk_sequence": idx,
                "token_count": len(chunk.split()),
                "accessible_roles": doc["accessible_roles"],
                "content": chunk,
            }
        )

if Path("artifacts").exists() or not (Path("..") / "artifacts").exists():
    artifacts_base = Path("artifacts") / "module2"
else:
    artifacts_base = Path("..") / "artifacts" / "module2"

chunks_dir = artifacts_base / "chunks"
metadata_dir = artifacts_base / "metadata"
chunks_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

chunks_csv_path = chunks_dir / "cleaned_formatted_chunks.csv"
chunks_jsonl_path = chunks_dir / "cleaned_formatted_chunks.jsonl"
role_mapping_csv_path = metadata_dir / "role_chunk_mapping.csv"
role_mapping_md_path = metadata_dir / "role_metadata_mapping.md"

chunk_fields = [
    "chunk_id",
    "source_document",
    "source_path",
    "department",
    "doc_type",
    "chunk_sequence",
    "token_count",
    "accessible_roles",
    "content",
]

with chunks_csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=chunk_fields)
    writer.writeheader()
    for row in chunk_records:
        csv_row = dict(row)
        csv_row["accessible_roles"] = ",".join(row["accessible_roles"])
        writer.writerow(csv_row)

with chunks_jsonl_path.open("w", encoding="utf-8") as f:
    for row in chunk_records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

mapping_fields = ["chunk_id", "source_document", "department", "accessible_roles"]
with role_mapping_csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=mapping_fields)
    writer.writeheader()
    for row in chunk_records:
        writer.writerow(
            {
                "chunk_id": row["chunk_id"],
                "source_document": row["source_document"],
                "department": row["department"],
                "accessible_roles": ",".join(row["accessible_roles"]),
            }
        )

dept_roles = {}
for row in chunk_records:
    dept_roles.setdefault(row["department"], row["accessible_roles"])

lines = [
    "# Role-Based Metadata Mapping",
    "",
    "This document links each chunk to source metadata and role permissions.",
    "",
    "## Department to Roles",
    "",
]
for dept in sorted(dept_roles):
    lines.append(f"- **{dept}** -> {', '.join(dept_roles[dept])}")

lines.extend(["", "## Chunk Coverage by Document", ""])
coverage = {}
for row in chunk_records:
    coverage.setdefault(row["department"], {})
    coverage[row["department"]].setdefault(row["source_document"], 0)
    coverage[row["department"]][row["source_document"]] += 1

for dept in sorted(coverage):
    lines.append(f"### {dept}")
    for source_doc, count in sorted(coverage[dept].items()):
        lines.append(f"- {source_doc}: {count} chunk(s)")
    lines.append("")

role_mapping_md_path.write_text("\n".join(lines).strip() + "\n", encoding="utf-8")

print(f"Documents: {len(raw_docs)}")
print(f"Chunks: {len(all_chunks)}")
print(f"Saved chunk CSV: {chunks_csv_path.resolve()}")
print(f"Saved chunk JSONL: {chunks_jsonl_path.resolve()}")
print(f"Saved role mapping CSV: {role_mapping_csv_path.resolve()}")
print(f"Saved role mapping markdown: {role_mapping_md_path.resolve()}")
print("\nFirst chunk preview:\n")
print(all_chunks[0][:500] if all_chunks else "No chunks generated.")

Documents: 10
Chunks: 311
Saved chunk CSV: D:\Projects\RBAC\artifacts\module2\chunks\cleaned_formatted_chunks.csv
Saved chunk JSONL: D:\Projects\RBAC\artifacts\module2\chunks\cleaned_formatted_chunks.jsonl
Saved role mapping CSV: D:\Projects\RBAC\artifacts\module2\metadata\role_chunk_mapping.csv
Saved role mapping markdown: D:\Projects\RBAC\artifacts\module2\metadata\role_metadata_mapping.md

First chunk preview:

finsolve technologies engineering document  1. introduction  1.1 company overview finsolve technologies is a leading fintech company headquartered in bangalore, india, with operations across north america, europe, and asia-pacific. founded in 2018, finsolve provides innovative financial solutions, including digital banking, payment processing, wealth management, and enterprise financial analytics,
